# Basis Size and FB/FS Ratio

**Sweep:** `basis_size`  
**Question:** How large must the total basis be? What mixture of Fourier-Bessel (FB) and fundamental-solution (FS) functions gives the lowest eigenvalue error?

**Sweep variables:**
- `n_basis` ∈ {20, 40, 80, 160, 320, 640}
- `fb_fraction` ∈ {0.0, 0.25, 0.5, 0.75, 1.0} — fraction of basis that is FB

**Domains:** rect, L_shape, iso_right_tri, eq_tri, disk_sector, GWW1 (all have closed-form reference eigenvalues)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd
import nb_utils

nb_utils.set_publication_style()
RESULTS_DIR = os.path.abspath('../results')

df, raw = nb_utils.load_sweep('basis_size', RESULTS_DIR)
print(f'Loaded {len(df)} results')
print('Domains:', df['domain_name'].unique())
print('n_basis values:', sorted(df['n_basis'].unique()))
print('fb_fraction values:', sorted(df['fb_fraction'].unique()))
df.head(3)

## Plot 1: Convergence with Basis Size

Using the balanced `fb_fraction = 0.5` mix, how does the maximum relative eigenvalue error decay as the basis grows?

In [ ]:
domains = sorted(df['domain_name'].unique())
colors = nb_utils.domain_color_map(domains)

sub = df[df['fb_fraction'].round(3) == 0.5].copy()

fig, ax = plt.subplots(figsize=(7, 4))
for dom in domains:
    d = sub[sub['domain_name'] == dom].sort_values('n_basis')
    if d['max_rel_error'].notna().any():
        ax.loglog(d['n_basis'], d['max_rel_error'],
                  marker='o', color=colors[dom],
                  label=nb_utils.label_domain(dom))

nb_utils.accuracy_threshold_line(ax, 1e-10, label='1e-10 accuracy')
ax.set_xlabel('Total basis size  $n_{\\mathrm{basis}}$')
ax.set_ylabel('Max relative eigenvalue error')
ax.set_title('Convergence with basis size  (FB fraction = 0.5)')
ax.legend(loc='lower left', fontsize=8)
plt.tight_layout()
plt.show()

## Plot 2: Effect of FB/FS Fraction per Domain

Does the optimal FB/FS mix vary by domain type? Each panel shows all five fraction values.

In [ ]:
fracs = sorted(df['fb_fraction'].round(3).unique())
frac_cmap = cm.get_cmap('plasma', len(fracs))
frac_colors = {f: frac_cmap(i / (len(fracs) - 1)) for i, f in enumerate(fracs)}

fig, axes = plt.subplots(2, 3, figsize=(11, 7), sharey=False)
axes = axes.flatten()

for idx, dom in enumerate(domains):
    ax = axes[idx]
    d_dom = df[df['domain_name'] == dom]
    for frac in fracs:
        d = d_dom[d_dom['fb_fraction'].round(3) == frac].sort_values('n_basis')
        if d['max_rel_error'].notna().any():
            ax.semilogy(d['n_basis'], d['max_rel_error'],
                        marker='o', markersize=4,
                        color=frac_colors[frac],
                        label=f'{int(frac*100)}% FB')
    ax.set_title(nb_utils.label_domain(dom), fontsize=9)
    ax.set_xlabel('$n_{\\mathrm{basis}}$', fontsize=9)
    ax.set_ylabel('Max rel. error', fontsize=9)

# Shared legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5,
           fontsize=9, bbox_to_anchor=(0.5, -0.03))
fig.suptitle('Effect of FB/FS ratio on accuracy', fontsize=11)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

## Plot 3: Optimal FB Fraction Heatmap

For each (domain, n_basis) pair, which fb_fraction achieves the lowest `max_rel_error`?

In [ ]:
nb_vals = sorted(df['n_basis'].unique())
best = (df[df['max_rel_error'].notna()]
        .loc[df.groupby(['domain_name', 'n_basis'])['max_rel_error'].idxmin()]
        [['domain_name', 'n_basis', 'fb_fraction']])

pivot = best.pivot(index='domain_name', columns='n_basis', values='fb_fraction')

fig, ax = plt.subplots(figsize=(8, 3.5))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn',
               vmin=0, vmax=1, origin='upper')

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([nb_utils.label_domain(d) for d in pivot.index], fontsize=9)
ax.set_xlabel('Total basis size')
ax.set_title('Optimal FB fraction (minimises max relative error)')

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('FB fraction')
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])

# Annotate cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## Plot 4: Wall Time vs Basis Size

How does computation time scale with basis size? (fb_fraction = 0.5)

In [ ]:
sub = df[df['fb_fraction'].round(3) == 0.5].copy()

fig, ax = plt.subplots(figsize=(7, 4))
for dom in domains:
    d = sub[sub['domain_name'] == dom].sort_values('n_basis')
    ax.plot(d['n_basis'], d['wall_time'],
            marker='o', color=colors[dom],
            label=nb_utils.label_domain(dom))

ax.set_xlabel('Total basis size  $n_{\\mathrm{basis}}$')
ax.set_ylabel('Wall time (s)')
ax.set_title('Computation time vs basis size  (FB fraction = 0.5)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Plot 5: Per-Eigenvalue Errors at n_basis = 160

Do higher-index eigenvalues converge more slowly? Each line is a different fb_fraction.

In [ ]:
TARGET_N = 160

# Build index: (domain_name, n_basis, fb_fraction) → raw result
index = {}
for r in raw:
    cfg = r.config
    nb = cfg.n_fb + cfg.n_fs
    frac = round(cfg.n_fb / nb, 3) if nb > 0 else 0
    index[(cfg.domain_name, nb, frac)] = r

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
axes = axes.flatten()

for idx, dom in enumerate(domains):
    ax = axes[idx]
    for frac in fracs:
        r = index.get((dom, TARGET_N, frac))
        if r is None or r.rel_errors is None:
            continue
        eig_idx = np.arange(1, len(r.rel_errors) + 1)
        ax.semilogy(eig_idx, r.rel_errors,
                    marker='o', markersize=4,
                    color=frac_colors[frac],
                    label=f'{int(frac*100)}% FB')
    ax.set_title(nb_utils.label_domain(dom), fontsize=9)
    ax.set_xlabel('Eigenvalue index', fontsize=9)
    ax.set_ylabel('Relative error', fontsize=9)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=5,
           fontsize=9, bbox_to_anchor=(0.5, -0.03))
fig.suptitle(f'Per-eigenvalue relative errors  (n_basis = {TARGET_N})', fontsize=11)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

## Plot 6: Accuracy vs Wall Time (Pareto View)

The full accuracy-cost trade-off across all domains and basis sizes. Marker size encodes `n_basis`.

In [ ]:
sub = df[df['max_rel_error'].notna()].copy()

fig, ax = plt.subplots(figsize=(7, 4.5))
for dom in domains:
    d = sub[sub['domain_name'] == dom]
    sizes = 20 + 60 * (d['n_basis'] - d['n_basis'].min()) / (d['n_basis'].max() - d['n_basis'].min() + 1)
    ax.scatter(d['wall_time'], d['max_rel_error'],
               s=sizes, color=colors[dom], alpha=0.7,
               label=nb_utils.label_domain(dom))

ax.set_yscale('log')
ax.set_xlabel('Wall time (s)')
ax.set_ylabel('Max relative eigenvalue error')
ax.set_title('Accuracy vs computation cost  (point size ∝ n_basis)')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()